# 📖 What Are Agent Evals?

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Understand** what agent evals are and why they matter
2. **Identify** differences between evals and traditional testing
3. **Know** when to use each type of eval

---

## ⏱️ Time Estimate

**~25 minutes**

## 🧠 Theory: Why Agent Evals?

### The Problem with Traditional Testing

```
┌─────────────────────────────────────────────────────────────┐
│         TRADITIONAL UNIT TEST                                 │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  def test_add():
│      assert add(2, 3) == 5  # ALWAYS works                   │
│                                                             │
│  ✓ Deterministic                                          │
│  ✓ Reproducible                                           │
│  ✓ One answer possible                                    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Why Agents Are Different

```
┌─────────────────────────────────────────────────────────────┐
│              AGENT RESPONSE                                 │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  agent.ask("What's 2+2?")                                │
│                                                             │
│  Possible responses:                                       │
│  • "4"                                                     │
│  • "The answer is 4"                                      │
│  • "4, simple arithmetic"                                 │
│  • "Two plus two equals four"                              │
│  • "I\'m not sure what you mean" (wrong!)                  │
│  • "5" (hallucination!)                                   │
│                                                             │
│  ✗ Probabilistic                                         │
│  ✗ Multiple valid answers                                 │
│  ✗ Can hallucinate                                      │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 📊 What Are Agent Evals?

**Agent evals** are systematic ways to evaluate if an agent is:

1. **Correct** - Does it give right answers?
2. **Efficient** - Does it use right tools?
3. **Safe** - Doesn't hallucinate or expose harmful info?
4. **Helpful** - Does it solve the user's problem?

### Analogy: Code Review = Agent Evals

```
┌─────────────────────────────────────────────────────────────┐
│         CODE REVIEW ⟹ AGENT EVALS                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Senior Dev reviews Junior Dev's code:                   │
│  • Does it work? (output correctness)                     │
│  • Is it efficient? (tool usage)                          │
│  • Are there bugs? (errors)                              │
│  • Is it readable? (reasoning clarity)                    │
│                                                             │
│  ⇓                                                         │
│                                                             │
│  Evaluator reviews Agent's output:                           │
│  • Is answer correct? (LLM-as-judge)                     │
│  • Used right tools? (trajectory check)                  │
│  • Hallucinating? (fact check)                         │
│  • Reasoning sound? (trace review)                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘```

## 🔑 Core Concept: Eval vs Test

| Aspect | Traditional Test | Agent Eval |
|--------|------------------|-------------|
**| Determinism | ✅ Always deterministic | ❌ Probabilistic |
**| Answer | Exact match | Flexible matching |
**| Single run | Meaningful | ❌ Need multiple runs |
**| Failure | Clear error | Nuanced (maybe wrong?) |
**| Tool usage | N/A | Must check correct tool |

**Key insight**: An eval may pass 80% of the time and still be unreliable! Run lots of cases!

## 📦 Types of Agent Evals

```
┌─────────────────────────────────────────────────────────────┐
│              TYPES OF EVALS                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. OUTPUT EVALUATION                                      │
│     - Is the final answer correct?                         │
│     - Format check                                       │
│     - Is it helpful?                                     │
│                                                             │
│  2. TRAJECTORY EVALUATION                                 │
│     - Did it use the right path?                         │
│     - Were tools correct?                                 │
│     - How many steps?                                   │
│                                                             │
│  3. RAG EVALUATION (for knowledge retrieval)             │
│     - Did it retrieve correct info?                        │
│     - Did it cite sources?                                │
│                                                             │
│  4. SAFETY EVALUATION                                    │
│     - No harmful outputs?                                │
│     - No hallucinations?                                  │
│     - No private info leaked?                           │
│                                                             │
└─────────────────────────────────────────────────────────────┘```

## 💻 Simple Eval Example

Let's see how evals work in code:

In [ ]:
"""Simple agent eval example"""
from openai import OpenAI
import os

client = OpenAI()

# Our "agent" (simplified)
def simple_agent(query: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}]
    )
    return response.choices[0].message.content

# Test case
test_input = "What is the capital of France?"
expected_answer = "Paris"

# Run agent
result = simple_agent(test_input)
print(f"Input: {test_input}")
print(f"Expected: {expected_answer}")
print(f"Got: {result}")

# Simple eval
def evaluate_output(response: str, expected: str) -> dict:
    """Simple eval with fuzzy matching."""
    response_lower = response.lower()
    expected_lower = expected.lower()
    
    # Check if expected is in response or response in expected
    correct = expected_lower in response_lower or \
              any(word in response_lower for word in expected_lower.split())
    return {
        "passed": correct,
        "response": response,
        "reason": "Contains 'Paris'" if correct else "Missing expected answer"
    }

eval_result = evaluate_output(result, expected_answer)
print(f"\n📊 Eval Result:")
print(f"  Passed: {eval_result['passed']}")
print(f"  Reason: {eval_result['reason']}")

## 🧪 Why Multiple Test Cases Matter

In [ ]:
# Run 5 test cases for "capital of X"
test_cases = [
    ("What is the capital of France?", "Paris"),
    ("What's the capital of Japan?", "Tokyo"),
    ("Capital of Germany?", "Berlin"),
    ("Tell me the capital city of Italy", "Rome"),
    ("What's the capital of Spain?", "Madrid"),
]

results = []
for query, expected in test_cases:
    result = simple_agent(query)
    eval_result = evaluate_output(result, expected)
    results.append({
        "query": query,
        "expected": expected,
        "got": result[:50],
        "passed": eval_result["passed"]
    })

print("📊 Multiple Test Results")
print("=" * 70)
for r in results:
    status = "✅" if r["passed"] else "❌"
    print(f"{status} Query: {r['query']}")
    print(f"   Expected: {r['expected']}")
    print(f"   Got: {r['got']}")
    print()

passed = sum(1 for r in results if r["passed"])
total = len(results)
print(f"📈 Overall: {passed}/{total} passed ({100*passed/total:.0f}%)")

---

## 🧠 Key Insights

1. **One test isn't enough** - Run many test cases!
2. **Fuzzy matching needed** - Same meaning, different words
3. **Trajectory matters** - How it got there matters
4. **Eval = not just pass/fail** - Understand WHY it failed

**Pro tip**: In production, aim for 95%+ pass rate with thousands of cases!

## ✅ Summary

You learned:
1. **Agent evals** evaluate if agents work correctly
2. **Different from tests** - probabilistic, fuzzy matching
3. **Multiple test cases** needed for confidence
4. **Output + trajectory** = full evaluation

## 🔗 Next Steps

Next: **[02_types_of_evals.ipynb](02_types_of_evals.ipynb)** - Deep dive into eval types!